## 0. Task

Binary court-judgment prediction, **macro-F1**. The holding is usually on the last page, so a 512-token head crop deletes the signal.


In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
SEARCH = [HERE, HERE.parent, Path("/kaggle/working")]
for base in (HERE, HERE.parent, HERE.parent.parent, HERE.parent.parent.parent):
    SEARCH.append(base)
    for phase in range(1, 8):
        SEARCH.append(base / f"phase{phase}")
        SEARCH.append(base / f"phase{phase}" / "code")
        SEARCH.append(base / "code" / f"phase{phase}")
for folder in SEARCH:
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
print("python path heads:")
for folder in sys.path[:8]:
    print(" ", folder)


## 1. Window packing

A case becomes several 256-token windows spread head→tail, with extra mass on the ending.


In [ ]:
import inspect
from phase4_own import windows_of, shorten, CaseClassifier, apply_method

print(inspect.getsource(windows_of))
print(inspect.getsource(shorten))


## 2. Classifier head + ending cue

Mean-pool window hidden states, two-way linear head. Last ~100 words also enter as a bag-of-words ending cue concatenated onto the logit.


In [ ]:
print(inspect.getsource(CaseClassifier.__init__))


## 3. Partial fine-tune (selected)

Unfreeze only the last transformer blocks of the efficient body so a T4 can hold a batch of windows. LoRMA / adapters / BitFit were the other PEFT zips.


In [ ]:
print(inspect.getsource(apply_method))
print("selected: --arch efficient --method partial")
print("script  : phase4_own.py")


## 4. Zip


In [ ]:
print("prediction.json: {case_id: 0 or 1}")
print("threshold chosen on dev macro-F1, not accuracy")
